In [1]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
from torch.nn import TransformerEncoder, TransformerEncoderLayer

In [36]:
num_files=15
start_time_frame = 1
file_prefix="/mnt/d/sources/cgan/standalone/dataset/latent_representation_for_"
all_dfs = []

for i in range(start_time_frame, num_files+1):
    print(f"{file_prefix}{i}.pkl.zip")
    df = pd.read_pickle(f"{file_prefix}{i}.pkl.zip", compression="zip")
    df = df[['x', 'y', 'z', 'time', 'latent_representation']]
    df['latent_representation'] = df['latent_representation'].apply(lambda x: x[0])
    all_dfs.append(df)

# Concatenate all dataframes
df_combined = pd.concat(all_dfs, ignore_index=True)
# Pivot table to get v as a matrix with shape (num_samples, 1200, 48)
df_pivot = df_combined.pivot_table(index=['x','y','z'], columns='time', values='latent_representation', aggfunc='first')
df_pivot_flat = df_pivot.reset_index()

/mnt/d/sources/cgan/standalone/dataset/latent_representation_for_1.pkl.zip
/mnt/d/sources/cgan/standalone/dataset/latent_representation_for_2.pkl.zip
/mnt/d/sources/cgan/standalone/dataset/latent_representation_for_3.pkl.zip
/mnt/d/sources/cgan/standalone/dataset/latent_representation_for_4.pkl.zip
/mnt/d/sources/cgan/standalone/dataset/latent_representation_for_5.pkl.zip
/mnt/d/sources/cgan/standalone/dataset/latent_representation_for_6.pkl.zip
/mnt/d/sources/cgan/standalone/dataset/latent_representation_for_7.pkl.zip
/mnt/d/sources/cgan/standalone/dataset/latent_representation_for_8.pkl.zip
/mnt/d/sources/cgan/standalone/dataset/latent_representation_for_9.pkl.zip
/mnt/d/sources/cgan/standalone/dataset/latent_representation_for_10.pkl.zip
/mnt/d/sources/cgan/standalone/dataset/latent_representation_for_11.pkl.zip
/mnt/d/sources/cgan/standalone/dataset/latent_representation_for_12.pkl.zip
/mnt/d/sources/cgan/standalone/dataset/latent_representation_for_13.pkl.zip
/mnt/d/sources/cgan/s

In [20]:
df_pivot_flat.head()

time,x,y,z,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,-117,-76,-25,"[0.60152996, 0.98103255, -0.008870157, -0.2454...","[0.5743986, 0.7089401, 0.5678579, -0.93600154,...","[-0.8775012, 0.77081597, 0.34914115, 0.9433937...","[-0.21666437, 0.276739, 0.67670155, -0.8597325...","[0.36412022, -0.76997614, 0.266663, -0.4527933...","[-0.3553231, -0.37004066, 0.40704593, -0.85804...","[0.4651682, 0.32191125, -0.8944155, 0.7075796,...","[-0.7426088, 0.7396835, -0.4338635, 0.6320076,...","[-0.16479653, 0.6672011, 0.44587672, -0.632495...","[-0.82881856, -0.4417005, -0.93003106, -0.8619...","[0.73687744, -0.4033748, -0.95862985, -0.47497...","[0.04219211, 0.5056695, -0.7162363, -0.4815756...","[-0.25694436, 0.4484407, -0.7467118, -0.507902...","[0.37512484, -0.87216157, -0.27912727, -0.6371...","[-0.8197715, -0.40555948, 0.9871446, -0.213071..."
1,-117,-76,-21,"[0.096365936, 0.54300946, 0.8387079, 0.4684101...","[-0.79866904, -0.12209187, 0.91154915, 0.76043...","[0.22371271, -0.7035653, -0.24191181, -0.04110...","[-0.12166012, -0.90468264, -0.703625, -0.35087...","[-0.6756259, 0.20125556, 0.41114074, -0.192610...","[-0.2619261, -0.9286362, 0.87315434, 0.1708212...","[0.12846749, -0.70874083, -0.09848327, 0.54393...","[-0.9192027, 0.39118025, 0.6679161, -0.7074012...","[0.75533044, -0.95989746, -0.90374166, -0.0630...","[0.23623419, 0.62348735, 0.38812354, 0.3865175...","[-0.7789519, 0.37796363, -0.87555295, -0.03135...","[0.7806267, 0.8975115, -0.5607745, -0.9473221,...","[0.95479465, 0.7803736, 0.1497075, -0.9292824,...","[-0.024801627, -0.3578735, -0.9100004, -0.1973...","[0.62667376, 0.6798731, 0.34012732, 0.5542102,..."
2,-117,-76,-17,"[0.6943935, 0.6303967, -0.8195155, 0.9442892, ...","[0.9876632, -0.61406004, -0.3469003, 0.4697731...","[-0.94120944, 0.9091235, 0.63393474, 0.7522126...","[0.9533979, 0.374611, 0.08358036, 0.45856717, ...","[0.103173874, -0.9596432, -0.21567442, 0.48207...","[0.8844934, -0.4611117, -0.35537854, -0.904041...","[0.19190139, -0.2850105, 0.09330799, 0.2111620...","[-0.55416745, 0.42543814, 0.32234558, -0.66634...","[0.6108569, 0.79253775, -0.007248677, 0.986103...","[-0.060510974, 0.8598963, -0.14712083, 0.79679...","[-0.16263914, 0.09215525, -0.3731518, 0.634308...","[-0.26758555, -0.94273233, 0.45466158, 0.84968...","[-0.60795283, -0.4063541, 0.5913332, -0.144027...","[-0.7996373, 0.7963824, 0.7067519, -0.98572326...","[0.8420596, 0.69856083, 0.15949044, 0.58392566..."
3,-117,-76,-13,"[0.56154877, 0.7658759, -0.31268513, 0.9745437...","[-0.7751982, -0.325407, 0.4743654, -0.13019444...","[0.31826624, 0.65888816, 0.98721373, -0.370199...","[-0.73936015, 0.13778922, -0.73885727, -0.4434...","[0.95708394, 0.9988378, -0.84009784, -0.483040...","[-0.43986663, -0.8859522, -0.60281193, -0.9730...","[0.628395, 0.42571187, -0.22475213, -0.5347901...","[0.8604549, -0.80334365, -0.19985121, 0.601780...","[0.62275165, 0.250471, -0.2056058, 0.48035592,...","[0.9358812, -0.9361252, -0.4325922, 0.56805587...","[0.5729953, 0.9068877, 0.5152233, 0.42906854, ...","[-0.7450801, 0.84187305, 0.71652573, -0.918943...","[-0.41168693, -0.66889644, 0.18859309, -0.2000...","[0.007128273, 0.85873616, 0.58662, -0.6822411,...","[0.17318805, -0.8831455, 0.7376071, 0.01548677..."
4,-117,-76,-9,"[0.9743955, -0.93269074, 0.027911449, 0.220510...","[-0.3710597, -0.19291241, 0.8153351, -0.286354...","[-0.9283217, 0.25671372, 0.90663964, 0.91436, ...","[-0.44492516, -0.2988802, -0.15240346, -0.3305...","[-0.5891063, -0.2228696, -0.18154225, -0.36551...","[0.70254093, 0.91327983, 0.54900235, 0.8643656...","[-0.8378765, -0.8759353, 0.020267947, -0.99677...","[-0.43921062, 0.9817179, 0.47819474, 0.8692322...","[0.975531, 0.67379576, 0.41823196, 0.63230884,...","[-0.6277772, 0.05303601, 0.62489265, -0.050239...","[0.97310853, -0.43587416, -0.16551034, -0.4941...","[0.66972435, 0.10993393, -0.88734424, 0.509057...","[0.17666648, 0.3667334, -0.90357393, -0.942890...","[-0.15619136, 0.62931603, -0.69244576, -0.7722...","[0.94355583, -0.54230475, 0.96

In [8]:
df_pivot_flat.shape

(33600, 13)

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [26]:
class SpatioTemporalDataset(Dataset):
    def __init__(self, dataframe, start_time_frame, sequence_length=5):
        self.dataframe = dataframe
        self.sequence_length = sequence_length
        self.start_time_frame = start_time_frame
        self.num_samples = len(self.dataframe)
        self.max_time_frame = self.dataframe.columns[-1]

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        if idx >= self.num_samples:
            raise IndexError("Index out of range")

        # Retrieve spatial coordinates
        coordinates = self.dataframe.iloc[idx][['x', 'y', 'z']].values.astype(np.float32)

        # Prepare the sequence of latent representations
        sequences = []
        num_time_window = (self.max_time_frame - self.start_time_frame) // (self.sequence_length)

        for t in range(1, num_time_window + 1):
            windows = []
            for c in range(t, t + self.sequence_length):
                time_col = c
                if time_col > self.max_time_frame:
                    raise IndexError(f"Time column {time_col} not found in the dataframe")

                vector = self.dataframe[int(time_col)].iloc[idx]
                windows.append(vector)

            windows = np.stack(windows)
            sequences.append(windows)
        sequences = np.stack(sequences)
        return coordinates, torch.tensor(sequences, dtype=torch.float)

In [10]:
idx = 3
coordinates = df_pivot_flat.iloc[idx][['x', 'y', 'z']].values.astype(np.float32)
coordinates

array([-117.,  -76.,  -13.], dtype=float32)

In [26]:
sequences = []
sequence_length=5
num_time_window = (15 - start_time_frame + 1) // (sequence_length)
num_time_window

2

In [28]:
# Prepare the sequence of latent representations

for t in range(0, num_time_window):
    print("T ",start_time_frame + t*sequence_length)
    t1= start_time_frame + t*sequence_length

    for c in range(t1, t1 + sequence_length):
        time_col = c
        print("c", c)
        # if time_col > self.max_time_frame:
        #     raise IndexError(f"Time column {time_col} not found in the dataframe")
        # 
        # vector = self.dataframe[int(time_col)].iloc[idx]

T  6
c 6
c 7
c 8
c 9
c 10
T  11
c 11
c 12
c 13
c 14
c 15


In [37]:
class SpatioTemporalDataset(Dataset):
    def __init__(self, dataframe, start_time_frame, sequence_length=5):
        self.dataframe = dataframe
        self.sequence_length = sequence_length
        self.start_time_frame = start_time_frame
        self.num_samples = len(self.dataframe)
        self.max_time_frame = self.dataframe.columns[-1]

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        if idx >= self.num_samples:
            raise IndexError("Index out of range")

        # Retrieve spatial coordinates
        coordinates = self.dataframe.iloc[idx][['x', 'y', 'z']].values.astype(np.float32)

        # Prepare the sequence of latent representations
        sequences = []
        num_time_window = (self.max_time_frame - self.start_time_frame + 1) // (self.sequence_length)

        for t in range(0, num_time_window):
            c_range = self.start_time_frame + t * self.sequence_length
            # windows = []
            for c in range(c_range, c_range + self.sequence_length):
                time_col = c
                if time_col > self.max_time_frame:
                    raise IndexError(f"Time column {time_col} not found in the dataframe")

                vector = self.dataframe[int(time_col)].iloc[idx]
                # windows.append(vector)
                sequences.append(vector)

            # windows = np.stack(windows)
            # sequences.append(windows)
        sequences = np.stack(sequences)
        return coordinates, torch.tensor(sequences, dtype=torch.float)

In [38]:
dataset = SpatioTemporalDataset(df_pivot_flat, 1,5)

In [39]:
a, b = dataset.__getitem__(0)

In [40]:
b.shape

torch.Size([15, 48])

In [24]:
a, b, c = dataset.__getitem__(0)

In [25]:
b.shape, c.shape

(torch.Size([14, 48]), torch.Size([1, 48]))

In [41]:
dataloader = DataLoader(dataset, batch_size=64, shuffle=True, drop_last=True)

In [48]:
i=0
for coord, seq in dataloader:
    print(seq.shape)
    print(len(dataloader))
    for seq1 in seq:
        print(seq1.shape)
        for i in range(0, seq1.shape[0], 5):
    # Extract 5 tensors at a time
            print(i)
            # tensors_slice = my_tensor[i:i+5]
        # for s in seq1:
        #     src_seq = s[:-1]
        #     tgt_seq = s[-1]
            # print(src_seq.shape)
            # print(tgt_seq.shape)
    if i==1:
        break
    i+=1




torch.Size([64, 15, 48])
525
torch.Size([15, 48])
0
5
10
torch.Size([15, 48])
0
5
10
torch.Size([15, 48])
0
5
10
torch.Size([15, 48])
0
5
10
torch.Size([15, 48])
0
5
10
torch.Size([15, 48])
0
5
10
torch.Size([15, 48])
0
5
10
torch.Size([15, 48])
0
5
10
torch.Size([15, 48])
0
5
10
torch.Size([15, 48])
0
5
10
torch.Size([15, 48])
0
5
10
torch.Size([15, 48])
0
5
10
torch.Size([15, 48])
0
5
10
torch.Size([15, 48])
0
5
10
torch.Size([15, 48])
0
5
10
torch.Size([15, 48])
0
5
10
torch.Size([15, 48])
0
5
10
torch.Size([15, 48])
0
5
10
torch.Size([15, 48])
0
5
10
torch.Size([15, 48])
0
5
10
torch.Size([15, 48])
0
5
10
torch.Size([15, 48])
0
5
10
torch.Size([15, 48])
0
5
10
torch.Size([15, 48])
0
5
10
torch.Size([15, 48])
0
5
10
torch.Size([15, 48])
0
5
10
torch.Size([15, 48])
0
5
10
torch.Size([15, 48])
0
5
10
torch.Size([15, 48])
0
5
10
torch.Size([15, 48])
0
5
10
torch.Size([15, 48])
0
5
10
torch.Size([15, 48])
0
5
10
torch.Size([15, 48])
0
5
10
torch.Size([15, 48])
0
5
10
torch.Size([15, 48]

KeyboardInterrupt: 

In [32]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import math

In [322]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)

        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * -(math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, 1, d_model)
        pe[:, 0, 0::2] = torch.sin(position * div_term)
        pe[:, 0, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:x.size(0), :]
        return self.dropout(x)
    
class TransformerModel(nn.Module):
    def __init__(self, input_size, ninp, nhead, nhid, nlayers, dropout=0.5):
        super(TransformerModel, self).__init__()
        from torch.nn import Transformer
        self.model_type = 'Transformer'
        self.pos_encoder = PositionalEncoding(ninp, dropout)
        self.input_fc = nn.Linear(input_size, ninp)
        self.transformer = Transformer(d_model=ninp, nhead=nhead,
                                       num_encoder_layers=nlayers,
                                       num_decoder_layers=nlayers,
                                       dim_feedforward=nhid, dropout=dropout)
        self.ninp = ninp
        self.decoder = nn.Linear(ninp, input_size)

    def forward(self, src, tgt):
        # print(src.shape, tgt.shape)
        src = self.input_fc(src) * math.sqrt(self.ninp)
        src = self.pos_encoder(src)
        tgt = self.input_fc(tgt) * math.sqrt(self.ninp)
        tgt = self.pos_encoder(tgt)
        # print("tgt", tgt.shape)
        # print("src", src.shape)
        output = self.transformer(src, tgt)
        # print(output.shape)
        
        output = self.decoder(output)
        return output


In [155]:
class Seq2PointPosTransformer(nn.Module):
    def __init__(self, nhead, num_encoder_layers, dim_feedforward, feature_size = 48, max_seq_len=5000, dropout=0.1):
        super(Seq2PointPosTransformer, self).__init__()

        self.embedding = nn.Linear(feature_size, feature_size)
        self.pos_encoder = PositionalEncoding(feature_size, max_len=max_seq_len)

        encoder_layer = nn.TransformerEncoderLayer(d_model=feature_size, nhead=nhead, dim_feedforward=dim_feedforward, dropout=dropout)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_encoder_layers)

        self.decoder = nn.Linear(feature_size, feature_size)

    def forward(self, src):
        src = self.embedding(src)
        src = self.pos_encoder(src)
        encoder_output = self.transformer_encoder(src)
        output = self.decoder(encoder_output[-1])  # Get the output of the last time step
        return torch.tanh(output)
    
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:x.size(0), :]
        return self.dropout(x)



In [310]:
class CustomTransformer(nn.Module):
    def __init__(self, input_size, num_heads, num_encoder_layers, num_decoder_layers, dim_feedforward):
        super(CustomTransformer, self).__init__()
        self.encoder_layer = nn.TransformerEncoderLayer(d_model=input_size, nhead=num_heads, dim_feedforward=dim_feedforward)
        self.transformer_encoder = nn.TransformerEncoder(self.encoder_layer, num_layers=num_encoder_layers)
        self.decoder_layer = nn.TransformerDecoderLayer(d_model=input_size, nhead=num_heads, dim_feedforward=dim_feedforward)
        self.transformer_decoder = nn.TransformerDecoder(self.decoder_layer, num_layers=num_decoder_layers)
        self.out = nn.Linear(input_size, input_size)

    def forward(self, src, tgt):
        memory = self.transformer_encoder(src)
        output = self.transformer_decoder(tgt, memory)
        return self.out(output)

In [311]:
input_size = 48  # Feature size
num_heads = 4    # Example value, you can modify
num_encoder_layers = 3  # Example value
num_decoder_layers = 3  # Example value
dim_feedforward = 2048  # Example value
epochs=300

In [312]:
model = CustomTransformer(input_size, num_heads, num_encoder_layers, num_decoder_layers, dim_feedforward)
optimizer = optim.SGD(model.parameters(), lr=0.01)
criterion = nn.MSELoss()

In [313]:
def train(model, dataloader, optimizer, criterion, epochs):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        i=0
        for coord, sequences in dataloader:
            # print(sequences.shape)
            i= i+1
            running_loss = 0
            for sequence in sequences:
                for s in sequence:
                    src_seq = s[:-1]
                    tgt_seq = s[-1].unsqueeze(0)
                    # print(src_seq.shape, tgt_seq.shape)

                    optimizer.zero_grad()
                    # print(src_seq.shape, tgt_seq.shape)
                    output = model(src_seq, tgt_seq)
                    # output = model(src_seq[-1])
                    loss = criterion(output, tgt_seq)
                    loss.backward()
                    optimizer.step()
                    running_loss +=loss.item()
            print(f'Sequence {i} /  Epoch {epoch}, Loss: {running_loss / len(sequences)}')
        total_loss += loss.item()
        print(f'Epoch {epoch}, Loss: {total_loss / len(dataloader)}')

In [314]:
train(model, dataloader, optimizer, criterion, epochs)

Sequence 1 /  Epoch 0, Loss: 1.4628984108567238
Sequence 2 /  Epoch 0, Loss: 1.233318682294339
Sequence 3 /  Epoch 0, Loss: 1.1869581607170403
Sequence 4 /  Epoch 0, Loss: 1.1835147377569228
Sequence 5 /  Epoch 0, Loss: 1.1561596733517945
Sequence 6 /  Epoch 0, Loss: 1.098255509044975
Sequence 7 /  Epoch 0, Loss: 1.0546900485642254
Sequence 8 /  Epoch 0, Loss: 0.9999392619356513
Sequence 9 /  Epoch 0, Loss: 0.9739618543535471
Sequence 10 /  Epoch 0, Loss: 0.9135275285225362
Sequence 11 /  Epoch 0, Loss: 0.8748115343041718
Sequence 12 /  Epoch 0, Loss: 0.8400456374511123
Sequence 13 /  Epoch 0, Loss: 0.8082136954180896
Sequence 14 /  Epoch 0, Loss: 0.7637379162479192
Sequence 15 /  Epoch 0, Loss: 0.7467175943311304
Sequence 16 /  Epoch 0, Loss: 0.7072874538134784
Sequence 17 /  Epoch 0, Loss: 0.6801791627658531
Sequence 18 /  Epoch 0, Loss: 0.6774286261061206
Sequence 19 /  Epoch 0, Loss: 0.6587561232736334
Sequence 20 /  Epoch 0, Loss: 0.6348526154179126
Sequence 21 /  Epoch 0, Loss: 0

KeyboardInterrupt: 

In [304]:
import torch
import torch.nn as nn
from torch.nn import TransformerEncoder, TransformerEncoderLayer


timeseries_length = 48 # Length of each time series
input_size = 192 # timeseries_length # Input size 
target_size = 48 # Target size
num_layers = 2 # Number of encoder layers
d_model = 128 # Encoder layer dimension
d_ff = 512 # Feed forward dimension
num_heads = 8 # Number of attention heads
dropout = 0.1

class TimeSeriesTransformer(nn.Module):

    def __init__(self):
        super().__init__()

        # Encoder 
        encoder_layer = TransformerEncoderLayer(d_model, num_heads, d_ff, dropout)
        self.encoder = TransformerEncoder(encoder_layer, num_layers)

        # Input projection 
        self.input_proj = nn.Linear(input_size, d_model)

        # Output projection
        self.output_proj = nn.Linear(d_model, target_size)

    def forward(self, src):
        # Project source input
        # print("src", src.shape)
        src = self.input_proj(src.view(-1).unsqueeze(0))
        # print("src1", src.shape)

        # Run through encoder 
        output = self.encoder(src)
        # print("output", output.shape)

        # Take the last embedded time step 
        output = output[-1]
        # print("output1", output.shape)

        # Project to target 
        output = self.output_proj(output)
        # print("output2", output.shape)

        return output


# src torch.Size([4, 48])
# src1 torch.Size([4, 128])
# output torch.Size([4, 128])
# output1 torch.Size([128])
# output2 torch.Size([1])

In [305]:
# Model parameters
num_timeseries = 5 # Number of time series 


In [306]:
model = TimeSeriesTransformer()
# model = TransformerModel(input_size=input_size, ninp=ninp, nhead=nhead, nhid=nhid, nlayers=nlayers, dropout=dropout)
# model = Seq2PointPosTransformer(nhead, num_encoder_layers=num_encoder_layers, dim_feedforward=dim_feedforward)
optimizer = optim.SGD(model.parameters(), lr=0.01)
criterion = nn.MSELoss()

In [307]:
def train(model, dataloader, optimizer, criterion, epochs):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        i=0
        l1 = 0
        for coord, sequences in dataloader:
            # print(sequences.shape)
            i= i+1
            l2 = 0
            running_loss = 0
            # print("dataloader", len(dataloader))
            for sequence in sequences:
                # print("sequences", len(sequences))
                l3=0
                for s in sequence:
                    # print("sequence", len(sequence))
                    src_seq = s[:-1]
                    tgt_seq = s[-1]

                    optimizer.zero_grad()
                    # print(src_seq.shape, tgt_seq.shape)
                    output = model(src_seq)
                    # output = model(src_seq[-1])
                    loss = criterion(output.view(-1), tgt_seq.view(-1))
                    loss.backward()
                    optimizer.step()
                    l3 +=loss.item()
                l2+= (l3 / len(sequence))
            l1 += (l2 / len(sequences))
            print(f'Sequence {i} /  Epoch {epoch}, Loss: {l2 / len(sequences)}')
            total_loss += l1
        print(f'Epoch {epoch}, Loss: {total_loss / len(dataloader)}')

In [308]:
train(model, dataloader, optimizer, criterion, epochs)

Sequence 1 /  Epoch 0, Loss: 0.6237525081572431
Sequence 2 /  Epoch 0, Loss: 0.5358938293841978
Sequence 3 /  Epoch 0, Loss: 0.4892453971939782
Sequence 4 /  Epoch 0, Loss: 0.4600284869472185
Sequence 5 /  Epoch 0, Loss: 0.4450600963706772
Sequence 6 /  Epoch 0, Loss: 0.4360073297284544
Sequence 7 /  Epoch 0, Loss: 0.436009302424888
Sequence 8 /  Epoch 0, Loss: 0.4321186061327657
Sequence 9 /  Epoch 0, Loss: 0.43673454970121384
Sequence 10 /  Epoch 0, Loss: 0.4180985664327939
Sequence 11 /  Epoch 0, Loss: 0.4198930716762939
Sequence 12 /  Epoch 0, Loss: 0.4183964516657093
Sequence 13 /  Epoch 0, Loss: 0.4178525434496502
Sequence 14 /  Epoch 0, Loss: 0.41930930782109505
Sequence 15 /  Epoch 0, Loss: 0.4059390591767927
Sequence 16 /  Epoch 0, Loss: 0.415178244933486
Sequence 17 /  Epoch 0, Loss: 0.41598999748627347
Sequence 18 /  Epoch 0, Loss: 0.41143848886713386
Sequence 19 /  Epoch 0, Loss: 0.40895990592737985
Sequence 20 /  Epoch 0, Loss: 0.4144641679401199
Sequence 21 /  Epoch 0, Lo

KeyboardInterrupt: 

In [323]:
input_size = 48
ninp=48
nhead=8
nhid=2048
nlayers = 6
dropout=0.1
epochs=300
feature_size = 48
num_encoder_layers = 2
num_decoder_layers = 2
dim_feedforward = 2048
# model = TransformerModel(d_model=input_size, nhead=nhead, num_encoder_layers=num_encoder_layers, num_decoder_layers=num_decoder_layers)
model = TransformerModel(input_size=input_size, ninp=ninp, nhead=nhead, nhid=nhid, nlayers=nlayers, dropout=dropout)
# model = Seq2PointPosTransformer(nhead, num_encoder_layers=num_encoder_layers, dim_feedforward=dim_feedforward)
optimizer = optim.SGD(model.parameters(), lr=0.01)
criterion = nn.MSELoss()

In [325]:


# def train(model, dataloader, optimizer, criterion, epochs):
#     model.train()
#     for epoch in range(epochs):
#         total_loss = 0
#         i=0
#         for coord, sequences in dataloader:
#             # print(sequences.shape)
#             i= i+1
#             running_loss = 0
#             for sequence in sequences:
#                 for s in sequence:
#                     src_seq = s[:-1]
#                     tgt_seq = s[-1]
# 
#                     optimizer.zero_grad()
#                     # print(src_seq.shape, tgt_seq.shape)
#                     output = model(src_seq[-1], tgt_seq)
#                     # output = model(src_seq[-1])
#                     loss = criterion(output[-1].view(-1), tgt_seq.view(-1))
#                     loss.backward()
#                     optimizer.step()
#                     running_loss +=loss.item()
#             print(f'Sequence {i} /  Epoch {epoch}, Loss: {running_loss / len(sequences)}')
#         total_loss += loss.item()
#         print(f'Epoch {epoch}, Loss: {total_loss / len(dataloader)}')

def train(model, dataloader, optimizer, criterion, epochs):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        i=0
        l1 = 0
        for coord, sequences in dataloader:
            # print(sequences.shape)
            i= i+1
            l2 = 0
            running_loss = 0
            # print("dataloader", len(dataloader))
            for sequence in sequences:
                # print("sequences", len(sequences))
                l3=0
                for s in sequence:
                    # print("sequence", len(sequence))
                    src_seq = s[:-1]
                    tgt_seq = s[-1]

                    optimizer.zero_grad()
                    # print(src_seq.shape, tgt_seq.shape)
                    output = model(src_seq[-1], tgt_seq)
                    # output = model(src_seq[-1])
                    loss = criterion(output[-1].view(-1), tgt_seq.view(-1))
                    loss.backward()
                    optimizer.step()
                    l3 +=loss.item()
                l2+= (l3 / len(sequence))
            l1 += (l2 / len(sequences))
            print(f'Sequence {i} /  Epoch {epoch}, Loss: {l2 / len(sequences)}')
            total_loss += l1
        print(f'Epoch {epoch}, Loss: {total_loss / len(dataloader)}')


In [326]:
train(model, dataloader, optimizer, criterion, epochs)

Sequence 1 /  Epoch 0, Loss: 0.4732697064367434
Sequence 2 /  Epoch 0, Loss: 0.42467299709096534
Sequence 3 /  Epoch 0, Loss: 0.409684547688812
Sequence 4 /  Epoch 0, Loss: 0.41362071068336576
Sequence 5 /  Epoch 0, Loss: 0.41264966068168485
Sequence 6 /  Epoch 0, Loss: 0.41139710306500393
Sequence 7 /  Epoch 0, Loss: 0.4089712697702149
Sequence 8 /  Epoch 0, Loss: 0.4032092737033963
Sequence 9 /  Epoch 0, Loss: 0.4019672049519917
Sequence 10 /  Epoch 0, Loss: 0.39554668605948495
Sequence 11 /  Epoch 0, Loss: 0.38479792761305964
Sequence 12 /  Epoch 0, Loss: 0.382133583150183
Sequence 13 /  Epoch 0, Loss: 0.3734894491887341
Sequence 14 /  Epoch 0, Loss: 0.36953280027955765
Sequence 15 /  Epoch 0, Loss: 0.364748662415271
Sequence 16 /  Epoch 0, Loss: 0.35581836166481196
Sequence 17 /  Epoch 0, Loss: 0.344383566795538
Sequence 18 /  Epoch 0, Loss: 0.33352970945027977
Sequence 19 /  Epoch 0, Loss: 0.3348782458342612
Sequence 20 /  Epoch 0, Loss: 0.3277709656395019
Sequence 21 /  Epoch 0, 

KeyboardInterrupt: 